**Aggregation and Grouping**

In [13]:
import pandas as pd
import sqlite3
customers_url= "https://raw.githubusercontent.com/graphql-compose/graphql-compose-examples/master/examples/northwind/data/csv/customers.csv"
orders_url="https://raw.githubusercontent.com/graphql-compose/graphql-compose-examples/master/examples/northwind/data/csv/orders.csv"
customers_df=pd.read_csv(customers_url)
orders_df=pd.read_csv(orders_url)

conn = sqlite3.connect(":memory:")
customers_df.to_sql("customers",conn,index=False,if_exists="replace")
orders_df.to_sql("orders",conn,index=False,if_exists="replace")

query1="""
SELECT
CustomerID,count(OrderID) as order_count,
sum(Freight) as total_freight,
avg(Freight) as avg_freight
FROM orders
GROUP BY CustomerID
ORDER BY total_freight DESC;
"""
print(pd.read_sql(query1,conn))

   customerID  order_count  total_freight  avg_freight
0       SAVEA           31        6683.70   215.603226
1       ERNSH           30        6205.39   206.846333
2       QUICK           28        5605.63   200.201071
3       HUNGO           19        2755.24   145.012632
4       RATTC           18        2134.21   118.567222
..        ...          ...            ...          ...
84      GALED            5          37.98     7.596000
85      NORTS            3          37.59    12.530000
86      LAZYK            2          19.40     9.700000
87      LAUGB            3           9.92     3.306667
88      CENTC            1           3.25     3.250000

[89 rows x 4 columns]


**WHERE vs. HAVING**

In [17]:
query2="""
SELECT CustomerID,count(Freight) as high_freight_orders
FROM orders
WHERE 'Freight' > 50
GROUP BY CustomerID;
"""
print(pd.read_sql(query2,conn))

   customerID  high_freight_orders
0       ALFKI                    6
1       ANATR                    4
2       ANTON                    7
3       AROUT                   13
4       BERGS                   18
..        ...                  ...
84      WARTH                   15
85      WELLI                    9
86      WHITC                   14
87      WILMK                    7
88      WOLZA                    7

[89 rows x 2 columns]


In [19]:
query3="""
SELECT CustomerID,SUM(Freight) as total_freight
FROM orders
GROUP BY CustomerID
HAVING SUM(Freight)> 500
ORDER BY total_freight DESC;
"""
print(pd.read_sql(query3,conn))

   customerID  total_freight
0       SAVEA        6683.70
1       ERNSH        6205.39
2       QUICK        5605.63
3       HUNGO        2755.24
4       RATTC        2134.21
5       QUEEN        1982.70
6       FOLKO        1678.08
7       BERGS        1559.52
8       FRANK        1403.44
9       MEREP        1394.22
10      BONAP        1357.87
11      WHITC        1353.06
12      HILAA        1259.16
13      PICCO        1186.11
14      GREAL        1087.61
15      LEHMS        1017.03
16      RICSU        1001.29
17      OLDWO         983.53
18      VAFFE         947.34
19      SEVES         913.81
20      OTTIK         862.74
21      EASTC         832.34
22      WARTH         822.48
23      SUPRD         821.23
24      KOENE         813.68
25      BOTTM         793.95
26      LILAS         734.41
27      HANAR         724.77
28      LINOD         673.81
29      FOLIG         637.94
30      LAMAI         635.82
31      RICAR         632.95
32      BLONP         623.66
33      GODOS 

**WHERE:**

where is used before aggreagation here only Freight with greater than 50 get counted

**HAVING:**

Having used to apply filter after aggregation here it gives the total of Freight is more than 500 is displayed


** JOIN and Aggregation**

In [29]:
query4="""
SELECT c.CompanyName,c.Country,count(o.OrderID) as order_count,sum(o.Freight) as total_freight
FROM customers c
INNER JOIN orders o ON c.CustomerID=o.CustomerID
GROUP BY C.CustomerID;
"""
result=pd.read_sql(query4,conn)
print(result)

                           companyName  country  order_count  total_freight
0                  Alfreds Futterkiste  Germany            6         225.58
1   Ana Trujillo Emparedados y helados   Mexico            4          97.42
2              Antonio Moreno Taquería   Mexico            7         268.52
3                      Around the Horn       UK           13         471.95
4                   Berglunds snabbköp   Sweden           18        1559.52
..                                 ...      ...          ...            ...
84                      Wartian Herkku  Finland           15         822.48
85              Wellington Importadora   Brazil            9         194.71
86                White Clover Markets      USA           14        1353.06
87                         Wilman Kala  Finland            7          88.41
88                      Wolski  Zajazd   Poland            7         175.74

[89 rows x 4 columns]


In [35]:
query5="""
SELECT c.CompanyName,c.Country,count(o.OrderID) as order_count,sum(o.Freight) as total_freight
FROM customers c
LEFT JOIN orders o ON c.CustomerID=o.CustomerID
GROUP BY C.CustomerID
ORDER BY total_freight DESC;
"""
result=pd.read_sql(query5,conn)
print(result.tail(5))

                             companyName country  order_count  total_freight
86                  Lazy K Kountry Store     USA            2          19.40
87         Laughing Bacchus Wine Cellars  Canada            3           9.92
88            Centro comercial Moctezuma  Mexico            1           3.25
89                     Paris spécialités  France            0            NaN
90  FISSA Fabrica Inter. Salchichas S.A.   Spain            0            NaN


**INNER JOIN:**

In inner join it gives all the required columns in both tables who has order in the other table by the customer id

**LEFT JOIN:**

In left join it gives all the values from the left table even with no order in the order table that give **0 and NaN for values are not in the table**
